# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashmita-21/Flyrank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes

The action queue reuses the repository's existing baseline scores and full-data model predictions. I do not train a new model or change the validated scoring formula in this notebook.

The final review score combines 70% model decline probability and 30% normalized baseline score. Items are ranked by this score, with impressions and sessions used as tie-breakers.

Each row receives reason codes that explain why it was selected. These include model decline risk, visible opportunity, stale content, low CTR, thin content, and low engagement where the corresponding repository rules are met.

I also assign a practical review archetype from the reason codes. These archetypes are operational categories for routing human review; they are not semantic clusters or independently validated labels.

| Review archetype | Evidence / reason | Suggested human action |
|---|---|---|
| Stale visible content | Old update date and meaningful impressions | Check accuracy, freshness, and whether a refresh is justified |
| Declining content with demand | Declining trend with existing impressions | Diagnose the cause before editing |
| Thin visible content | Low word count with meaningful impressions | Review whether important user needs are missing |
| CTR opportunity | Visible page, reasonable position, low CTR | Review title, snippet, and search intent |
| Engagement opportunity | Sufficient sessions with weak engagement signals | Review content usefulness and page experience |
| General monitoring | No stronger rule matched | Monitor; do not force an edit |

The queue is a prioritization aid, not an instruction to change every page. A high score means a page deserves earlier human inspection, not that a refresh will necessarily improve performance.


In [1]:
# Section 1: Build the Week 7 ranked action queue.
# Reuses the repository's official preparation, baseline, and model scripts.

import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

# 1. Locate the repository
candidates = [
    Path.cwd(),
    Path("/content/Flyrank-ML-Internship"),
    Path("/content"),
]

REPO_DIR = next(
    (
        p for p in candidates
        if (p / "scripts" / "01_prepare_features.py").exists()
    ),
    None
)

if REPO_DIR is None:
    REPO_DIR = Path("/content/Flyrank-ML-Internship")
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/rashmita-21/Flyrank-ML-Internship.git",
            str(REPO_DIR),
        ],
        check=True,
    )

sys.path.insert(0, str(REPO_DIR / "scripts"))

from ml_utils import (
    PROCESSED_DIR,
    OUTPUT_DIR,
    normalize,
    read_json,
)

# 2. Run the repository's official scripts to ensure source outputs exist
for script_name in [
    "01_prepare_features.py",
    "02_baseline_score.py",
    "03_train_model.py",
]:
    subprocess.run(
        [sys.executable, str(REPO_DIR / "scripts" / script_name)],
        cwd=REPO_DIR,
        check=True,
    )

# 3. Load the official pipeline outputs
feature_path = PROCESSED_DIR / "refresh_feature_vector.csv"
baseline_path = PROCESSED_DIR / "baseline_refresh_queue.csv"
prediction_path = PROCESSED_DIR / "model_predictions.csv"
results_path = OUTPUT_DIR / "model_results.json"

feature_frame = pd.read_csv(feature_path)
baseline_frame = pd.read_csv(baseline_path)
prediction_frame = pd.read_csv(prediction_path)
model_results = read_json(results_path)

required_prediction_columns = {
    "content_id",
    "best_model_name",
    "best_model_probability",
}

missing = required_prediction_columns - set(prediction_frame.columns)
if missing:
    raise ValueError(f"Missing prediction columns: {sorted(missing)}")

if feature_frame.empty or baseline_frame.empty or prediction_frame.empty:
    raise ValueError("One or more pipeline input files are empty.")

# 4. Reproduce the official Week 4/5/6 queue construction
final_frame = baseline_frame.merge(
    prediction_frame[
        [
            "content_id",
            "best_model_name",
            "best_model_probability",
            "prob_logistic_regression",
            "prob_decision_tree",
            "prob_random_forest",
        ]
    ],
    on="content_id",
    how="left",
    validate="one_to_one",
)

context_columns = [
    "content_id",
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

available_context = [
    col for col in context_columns if col in feature_frame.columns
]

final_frame = final_frame.merge(
    feature_frame[available_context],
    on="content_id",
    how="left",
    validate="one_to_one",
)

# A missing model probability should not silently become a valid low-risk score.
if final_frame["best_model_probability"].isna().any():
    missing_count = int(final_frame["best_model_probability"].isna().sum())
    raise ValueError(
        f"{missing_count} rows have no model probability. "
        "Check the content_id join before creating the queue."
    )

# Match the official score formula
final_frame["baseline_score_normalized"] = normalize(
    final_frame["baseline_refresh_score"]
)

final_frame["final_refresh_score"] = (
    100
    * (
        0.70 * final_frame["best_model_probability"]
        + 0.30 * final_frame["baseline_score_normalized"]
    )
).clip(0, 100)


# 5. Reproduce the official final reason-code rules
def merged_reason_codes(row):
    reasons = [
        reason
        for reason in str(row.get("reason_codes", "")).split("|")
        if reason and reason != "nan"
    ]

    if row["best_model_probability"] >= 0.65:
        reasons.append("model_decline_risk")

    if (
        row["best_model_probability"] >= 0.50
        and row["impressions_90d"] >= 500
    ):
        reasons.append("visible_model_opportunity")

    if (
        row["impressions_90d"] >= 500
        and row["avg_position"] > 0
        and row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("ctr_review_candidate")

    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("engagement_review_candidate")

    unique_reasons = list(dict.fromkeys(reasons))
    return "|".join(unique_reasons or ["general_refresh_review"])


def suggested_action(row):
    reasons = set(str(row["final_reason_codes"]).split("|"))

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"

    if (
        "ctr_review_candidate" in reasons
        and (
            "model_decline_risk" in reasons
            or "declining_with_demand" in reasons
        )
    ):
        return "refresh_and_review_ctr"

    if (
        "engagement_review_candidate" in reasons
        and (
            "model_decline_risk" in reasons
            or "declining_with_demand" in reasons
        )
    ):
        return "refresh_and_review_engagement"

    if {
        "model_decline_risk",
        "declining_with_demand",
        "stale_visible_page",
        "visible_model_opportunity",
    }.intersection(reasons):
        return "refresh"

    return "monitor"


final_frame["final_reason_codes"] = final_frame.apply(
    merged_reason_codes,
    axis=1,
)

final_frame["suggested_action"] = final_frame.apply(
    suggested_action,
    axis=1,
)

# 6. Reproduce the official confidence thresholds
high_threshold = float(
    final_frame["final_refresh_score"].quantile(0.80)
)
medium_threshold = float(
    final_frame["final_refresh_score"].quantile(0.50)
)


def confidence_label(row):
    if (
        row["final_refresh_score"] >= high_threshold
        and row["impressions_90d"] >= 500
        and row["sessions_90d"] >= 10
        and row["best_model_probability"] >= 0.50
    ):
        return "high"

    if row["final_refresh_score"] >= medium_threshold:
        return "medium"

    return "low"


final_frame["confidence"] = final_frame.apply(
    confidence_label,
    axis=1,
)

# 7. Add transparent operational archetypes.
# These are rule-based review categories, not ML clusters.
def assign_archetype(row):
    reasons = set(str(row["final_reason_codes"]).split("|"))

    if "thin_visible_page" in reasons:
        return "thin_visible_content"

    if "ctr_review_candidate" in reasons:
        return "ctr_opportunity"

    if "engagement_review_candidate" in reasons:
        return "engagement_opportunity"

    if "stale_visible_page" in reasons:
        return "stale_visible_content"

    if "declining_with_demand" in reasons or "model_decline_risk" in reasons:
        return "declining_content"

    if "page_one_decay_risk" in reasons:
        return "page_one_decay_risk"

    return "general_monitoring"


final_frame["review_archetype"] = final_frame.apply(
    assign_archetype,
    axis=1,
)

# 8. Rank exactly as the repository's official exporter does
final_frame = final_frame.sort_values(
    [
        "final_refresh_score",
        "impressions_90d",
        "sessions_90d",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

final_frame["final_rank"] = np.arange(1, len(final_frame) + 1)

# 9. Create the Week 7 output folder and ranked queue
WEEK7_OUTPUT_DIR = REPO_DIR / "work" / "outputs"
WEEK7_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_columns = [
    "final_rank",
    "content_id",
    "client_id",
    "final_refresh_score",
    "best_model_name",
    "best_model_probability",
    "baseline_refresh_score",
    "confidence",
    "review_archetype",
    "suggested_action",
    "final_reason_codes",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

missing_output_columns = [
    col for col in output_columns
    if col not in final_frame.columns
]

if missing_output_columns:
    raise ValueError(
        f"Required queue columns missing: {missing_output_columns}"
    )

ranked_queue = final_frame[output_columns].copy()

queue_path = WEEK7_OUTPUT_DIR / "ranked_action_queue.csv"
ranked_queue.to_csv(queue_path, index=False)

# 10. Sanity checks and summary
assert ranked_queue["final_rank"].is_monotonic_increasing
assert ranked_queue["content_id"].is_unique
assert ranked_queue["final_refresh_score"].between(0, 100).all()

print("Ranked action queue created:")
print(queue_path)
print(f"Rows in queue: {len(ranked_queue):,}")
print(f"Best model: {model_results['best_model']['name']}")
print(f"Model validation split: {model_results['split_strategy']}")
print(f"High-confidence rows: {(ranked_queue['confidence'] == 'high').sum():,}")

print("\nSuggested actions:")
display(ranked_queue["suggested_action"].value_counts().to_frame("rows"))

print("\nReview archetypes:")
display(ranked_queue["review_archetype"].value_counts().to_frame("rows"))

print("\nTop 10 ranked actions:")
display(ranked_queue.head(10))

Ranked action queue created:
/content/Flyrank-ML-Internship/work/outputs/ranked_action_queue.csv
Rows in queue: 30,000
Best model: random_forest
Model validation split: client_holdout
High-confidence rows: 3,602

Suggested actions:


,rows
suggested_action,
monitor,13083
refresh,8188
refresh_and_review_ctr,6654
refresh_and_review_engagement,1993
expand_and_refresh,82



Review archetypes:


,rows
review_archetype,
ctr_opportunity,9741
general_monitoring,8084
declining_content,6268
engagement_opportunity,3415
page_one_decay_risk,2405
thin_visible_content,82
stale_visible_content,5



Top 10 ranked actions:


,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,review_archetype,suggested_action,...,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.734212,random_forest,0.783472,0.844481,high,ctr_opportunity,refresh_and_review_ctr,...,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_d6570c51c9bd,client_3fdba35f04,81.603243,random_forest,0.849842,0.695884,medium,ctr_opportunity,refresh_and_review_ctr,...,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
2,3,content_6aa43079fb0c,client_3fdba35f04,81.544618,random_forest,0.789490,0.825477,high,ctr_opportunity,refresh_and_review_ctr,...,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
3,4,content_72e800a9c214,client_3fdba35f04,81.169731,random_forest,0.776297,0.842545,high,ctr_opportunity,refresh_and_review_ctr,...,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.957565,random_forest,0.816010,0.749468,medium,ctr_opportunity,refresh_and_review_ctr,...,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
5,6,content_b69288c5e701,client_3fdba35f04,80.798090,random_forest,0.796332,0.787358,high,ctr_opportunity,refresh_and_review_ctr,...,1370.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
6,7,content_9b6df29f7889,client_3fdba35f04,80.650656,random_forest,0.846499,0.673530,high,ctr_opportunity,refresh_and_review_ctr,...,1415.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,moderate,page_1
7,8,content_ba6f9dfcbca1,client_3fdba35f04,80.432641,random_forest,0.827496,0.708128,medium,declining_content,refresh,...,1362.0,down,HIGH,keyword article,transactional,91-180,91-180,1000-2000,good,page_3_5
8,9,content_4d76cdb3387b,client_3fdba35f04,80.428403,random_forest,0.844030,0.671993,medium,ctr_opportunity,refresh_and_review_ctr,...,1554.0,down,HIGH,keyword article,commercial,91-180,91-180,1000-2000,moderate,top_3
9,10,content_b4f35d640b1c,client_3fdba35f04,80.428243,random_forest,0.845325,0.669168,medium,declining_content,refresh,...,1389.0,down,HIGH,keyword article,commercial,91-180,91-180,1000-2000,good,page_3_5


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

The ranked action queue is a decision-support tool for helping content teams decide which existing pages deserve an earlier review.

The final refresh score combines the model's estimated decline probability with the repository's baseline refresh score. Higher-ranked pages should be inspected earlier, especially when their reason codes show visible traffic, declining performance, or a potential content-quality issue.

The queue is intended for:
- Prioritizing pages for editorial and SEO review.
- Explaining why a page was selected through reason codes.
- Identifying possible refresh, CTR, engagement, or monitoring opportunities.
- Supporting a human decision about whether an update is worthwhile.

### Limits

The score is not a probability that a refresh will succeed. It does not estimate the causal impact of editing a page, guarantee traffic recovery, or predict Google's ranking decisions.

The model was evaluated on historical data. Changes in client mix, search demand, seasonality, tracking, and search-engine behavior may reduce its usefulness on new data.

The queue also inherits the limitations of its input features and labels. In particular, temporal availability of performance features must be verified before treating the score as a prospective prediction.

The queue is non-production and must not automatically publish, edit, delete, redirect, or change the metadata of any page. Every proposed action requires human review.

In [2]:
# Section 2: Summarize intended use, score interpretation, and queue limits.

assert "ranked_queue" in globals(), (
    "Run Section 1 first to create ranked_queue."
)

# Show the queue's score distribution and review workload.
score_summary = ranked_queue["final_refresh_score"].describe().to_frame(
    "Final refresh score"
)

confidence_summary = (
    ranked_queue["confidence"]
    .value_counts(dropna=False)
    .rename_axis("Confidence")
    .to_frame("Pages")
)

action_summary = (
    ranked_queue["suggested_action"]
    .value_counts(dropna=False)
    .rename_axis("Suggested action")
    .to_frame("Pages")
)

print("Score distribution")
display(score_summary.round(2))

print("Confidence distribution")
display(confidence_summary)

print("Suggested action distribution")
display(action_summary)

# Document the score interpretation for reproducibility.
score_definition = {
    "score_name": "final_refresh_score",
    "formula": (
        "100 * (0.70 * best_model_probability "
        "+ 0.30 * baseline_score_normalized)"
    ),
    "interpretation": (
        "Relative prioritization score for human review; "
        "not probability of refresh success."
    ),
    "intended_use": (
        "Prioritize existing content pages for human review."
    ),
    "not_intended_for": [
        "Automatic publishing",
        "Automatic content editing",
        "Automatic deletion or redirection",
        "Guaranteeing traffic recovery",
        "Predicting Google ranking decisions",
    ],
    "review_required": True,
    "production_status": "Non-production decision-support prototype",
}

import json

metadata_path = WEEK7_OUTPUT_DIR / "action_playbook_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(score_definition, f, indent=2)

print("Action-playbook metadata saved to:")
print(metadata_path)

Score distribution


,Final refresh score
count,30000.00
mean,49.80
std,16.94
min,0.85
25%,41.25
50%,53.61
75%,61.78
max,81.73


Confidence distribution


,Pages
Confidence,
low,15000
medium,11398
high,3602


Suggested action distribution


,Pages
Suggested action,
monitor,13083
refresh,8188
refresh_and_review_ctr,6654
refresh_and_review_engagement,1993
expand_and_refresh,82


Action-playbook metadata saved to:
/content/Flyrank-ML-Internship/work/outputs/action_playbook_metadata.json


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Every page in the ranked queue must be reviewed by a person before an action is approved.

The reviewer should:

1. Open the page and confirm that it is still live and relevant.
2. Check the reason codes and verify that the underlying issue is visible in the source data.
3. Review recent impressions, clicks, sessions, CTR, and average position.
4. Check search intent, factual accuracy, content quality, and whether the page has a meaningful user need.
5. Check for seasonality, tracking changes, cannibalization, and other possible explanations for a performance decline.
6. Estimate the effort and likely value of an update.
7. Record the decision: refresh, investigate, monitor, or no action.

A high-confidence label means the row meets the queue's scoring and evidence thresholds. It does not remove the need for review. Low-confidence items should receive additional investigation before any action is considered.

### No-go list

The model must not independently:

- Publish or rewrite content.
- Delete pages or change URLs.
- Create redirects or modify canonical tags.
- Change titles, metadata, schema, or internal links.
- Make legal, medical, financial, or other sensitive editorial decisions.
- Treat a decline score as proof that a refresh will improve performance.
- Override subject-matter experts or client approval.

All changes remain subject to editorial, SEO, and client approval. The model only helps prioritize the review queue.

In [3]:
# Section 3: Create a human-review checklist and no-go policy.

import json
from pathlib import Path

assert "ranked_queue" in globals(), (
    "Run Section 1 before running this section."
)

review_checklist = [
    "Confirm the page is live and relevant.",
    "Verify the reason codes against the source data.",
    "Review impressions, clicks, sessions, CTR, and position.",
    "Check search intent, accuracy, and content quality.",
    "Investigate seasonality, tracking changes, and cannibalization.",
    "Estimate effort and potential business value.",
    "Record refresh, investigate, monitor, or no-action decision.",
]

no_go_actions = [
    "Automatic publishing or rewriting",
    "Automatic page deletion",
    "Automatic URL changes or redirects",
    "Automatic canonical, schema, or metadata changes",
    "Sensitive editorial decisions without expert review",
    "Treating decline risk as proof of refresh benefit",
    "Overriding client or subject-matter approval",
]

# Create a human-review status for each queue row.
review_queue = ranked_queue.copy()

review_queue["human_review_status"] = "pending_review"
review_queue["review_decision"] = "not_recorded"
review_queue["reviewer_notes"] = ""

# High score does not mean automatic approval.
review_queue["requires_human_approval"] = True

# Low-confidence items receive an additional investigation flag.
review_queue["additional_investigation"] = (
    review_queue["confidence"].eq("low")
)

review_policy = {
    "review_required_for_every_row": True,
    "high_confidence_means_automatic_approval": False,
    "low_confidence_requires_additional_investigation": True,
    "review_checklist": review_checklist,
    "no_go_actions": no_go_actions,
    "allowed_human_decisions": [
        "refresh",
        "investigate",
        "monitor",
        "no_action",
    ],
}

policy_path = WEEK7_OUTPUT_DIR / "human_review_policy.json"

with open(policy_path, "w", encoding="utf-8") as f:
    json.dump(review_policy, f, indent=2)

print("Human-review checklist:")
for item in review_checklist:
    print("-", item)

print("\nNo-go actions:")
for item in no_go_actions:
    print("-", item)

print("\nReview workload:")
display(
    review_queue["additional_investigation"]
    .value_counts()
    .rename_axis("Additional investigation required")
    .to_frame("Pages")
)

print("\nPolicy saved to:", policy_path)

Human-review checklist:
- Confirm the page is live and relevant.
- Verify the reason codes against the source data.
- Review impressions, clicks, sessions, CTR, and position.
- Check search intent, accuracy, and content quality.
- Investigate seasonality, tracking changes, and cannibalization.
- Estimate effort and potential business value.
- Record refresh, investigate, monitor, or no-action decision.

No-go actions:
- Automatic publishing or rewriting
- Automatic page deletion
- Automatic URL changes or redirects
- Automatic canonical, schema, or metadata changes
- Sensitive editorial decisions without expert review
- Treating decline risk as proof of refresh benefit
- Overriding client or subject-matter approval

Review workload:


,Pages
Additional investigation required,
False,15000
True,15000



Policy saved to: /content/Flyrank-ML-Internship/work/outputs/human_review_policy.json


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

This is a non-production prototype. If the queue is used in a review exercise, its usefulness should be checked periodically against subsequent content performance and reviewer decisions.

I would monitor:

- **Data quality:** missing values, duplicate content IDs, unexpected ranges, and changes in feature availability.
- **Score distribution:** major changes in the model probability or final refresh-score distribution.
- **Client and content mix:** changes in the number of clients, content types, and page categories.
- **Ranking usefulness:** whether high-ranked pages contain issues that reviewers consider actionable.
- **Outcome tracking:** where possible, compare reviewed pages with later performance, while accounting for seasonality and other changes.
- **Human feedback:** how often reviewers accept, reject, or defer recommendations.

### Review and retraining triggers

A model review should be considered when input schemas change, missingness increases, score distributions shift substantially, or reviewers repeatedly find that the top-ranked recommendations are not useful.

Retraining should only be considered after checking data quality, label consistency, feature timing, and the availability of enough new labelled examples. The updated model must be evaluated with an appropriate grouped or time-aware validation split before it replaces the existing model.

A change in traffic alone is not sufficient evidence that the model needs retraining. It may reflect seasonality, search demand, tracking changes, or client mix.

In [4]:
# Section 4: Create a lightweight monitoring snapshot and trigger policy.

import json
import numpy as np
import pandas as pd

assert "ranked_queue" in globals(), (
    "Run Section 1 before running this section."
)

# 1. Data-quality checks
monitoring_checks = {
    "queue_rows": int(len(ranked_queue)),
    "unique_content_ids": int(ranked_queue["content_id"].nunique()),
    "duplicate_content_ids": int(ranked_queue["content_id"].duplicated().sum()),
    "missing_model_probabilities": int(
        ranked_queue["best_model_probability"].isna().sum()
    ),
    "missing_final_scores": int(
        ranked_queue["final_refresh_score"].isna().sum()
    ),
    "probabilities_outside_0_1": int(
        (
            ~ranked_queue["best_model_probability"].between(0, 1)
        ).sum()
    ),
    "scores_outside_0_100": int(
        (
            ~ranked_queue["final_refresh_score"].between(0, 100)
        ).sum()
    ),
}

# 2. Score and queue distribution snapshot
score_monitoring = {
    "model_probability_mean": float(
        ranked_queue["best_model_probability"].mean()
    ),
    "model_probability_median": float(
        ranked_queue["best_model_probability"].median()
    ),
    "model_probability_p90": float(
        ranked_queue["best_model_probability"].quantile(0.90)
    ),
    "final_score_mean": float(
        ranked_queue["final_refresh_score"].mean()
    ),
    "final_score_median": float(
        ranked_queue["final_refresh_score"].median()
    ),
    "final_score_p90": float(
        ranked_queue["final_refresh_score"].quantile(0.90)
    ),
}

confidence_distribution = (
    ranked_queue["confidence"]
    .value_counts(normalize=True)
    .to_dict()
)

action_distribution = (
    ranked_queue["suggested_action"]
    .value_counts()
    .to_dict()
)

# 3. Practical triggers for human investigation
trigger_policy = {
    "data_quality": [
        "Investigate any duplicate content IDs.",
        "Investigate missing model probabilities or final scores.",
        "Review unexpected feature missingness or range changes.",
    ],
    "score_distribution": [
        "Review substantial changes in model probability or final-score distributions.",
        "Compare current distributions with a saved historical monitoring snapshot.",
    ],
    "ranking_usefulness": [
        "Review repeated low-value recommendations among the highest-ranked pages.",
        "Collect reviewer acceptance, rejection, and deferral rates.",
    ],
    "retraining": [
        "Consider retraining only after label and feature timing checks.",
        "Require enough new labelled examples.",
        "Re-evaluate using grouped or time-aware validation.",
        "Do not automatically replace the current model.",
    ],
    "important_caveat": (
        "This notebook creates a baseline monitoring snapshot. "
        "It does not establish statistical drift thresholds or run continuous monitoring."
    ),
}

monitoring_report = {
    "monitoring_checks": monitoring_checks,
    "score_distribution": score_monitoring,
    "confidence_distribution": confidence_distribution,
    "suggested_action_distribution": action_distribution,
    "trigger_policy": trigger_policy,
}

monitoring_path = WEEK7_OUTPUT_DIR / "monitoring_snapshot.json"

with open(monitoring_path, "w", encoding="utf-8") as f:
    json.dump(monitoring_report, f, indent=2)

print("Data-quality checks:")
display(
    pd.DataFrame(
        list(monitoring_checks.items()),
        columns=["Check", "Value"]
    )
)

print("\nScore distribution:")
display(
    pd.DataFrame(
        list(score_monitoring.items()),
        columns=["Metric", "Value"]
    ).round(4)
)

print("\nMonitoring snapshot saved to:", monitoring_path)
print(
    "\nNote: This is a one-time snapshot. "
    "A future monitoring run should compare it with a dated historical snapshot."
)

Data-quality checks:


,Check,Value
0,queue_rows,30000
1,unique_content_ids,30000
2,duplicate_content_ids,0
3,missing_model_probabilities,0
4,missing_final_scores,0
5,probabilities_outside_0_1,0
6,scores_outside_0_100,0



Score distribution:


,Metric,Value
0,model_probability_mean,0.5089
1,model_probability_median,0.5381
2,model_probability_p90,0.7238
3,final_score_mean,49.7995
4,final_score_median,53.6075
5,final_score_p90,68.1714



Monitoring snapshot saved to: /content/Flyrank-ML-Internship/work/outputs/monitoring_snapshot.json

Note: This is a one-time snapshot. A future monitoring run should compare it with a dated historical snapshot.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Monitoring and retrain triggers

This is a non-production prototype. If the queue is used in a review exercise, its usefulness should be checked periodically against subsequent content performance and reviewer decisions.

I would monitor:

- **Data quality:** missing values, duplicate content IDs, unexpected ranges, and changes in feature availability.
- **Score distribution:** major changes in the model probability or final refresh-score distribution.
- **Client and content mix:** changes in the number of clients, content types, and page categories.
- **Ranking usefulness:** whether high-ranked pages contain issues that reviewers consider actionable.
- **Outcome tracking:** where possible, compare reviewed pages with later performance, while accounting for seasonality and other changes.
- **Human feedback:** how often reviewers accept, reject, or defer recommendations.

### Review and retraining triggers

A model review should be considered when input schemas change, missingness increases, score distributions shift substantially, or reviewers repeatedly find that the top-ranked recommendations are not useful.

Retraining should only be considered after checking data quality, label consistency, feature timing, and the availability of enough new labelled examples. The updated model must be evaluated with an appropriate grouped or time-aware validation split before it replaces the existing model.

A change in traffic alone is not sufficient evidence that the model needs retraining. It may reflect seasonality, search demand, tracking changes, or client mix.

In [6]:
# Section 4: Create a lightweight monitoring snapshot and trigger policy.

import json
import numpy as np
import pandas as pd

assert "ranked_queue" in globals(), (
    "Run Section 1 before running this section."
)

# 1. Data-quality checks
monitoring_checks = {
    "queue_rows": int(len(ranked_queue)),
    "unique_content_ids": int(ranked_queue["content_id"].nunique()),
    "duplicate_content_ids": int(ranked_queue["content_id"].duplicated().sum()),
    "missing_model_probabilities": int(
        ranked_queue["best_model_probability"].isna().sum()
    ),
    "missing_final_scores": int(
        ranked_queue["final_refresh_score"].isna().sum()
    ),
    "probabilities_outside_0_1": int(
        (
            ~ranked_queue["best_model_probability"].between(0, 1)
        ).sum()
    ),
    "scores_outside_0_100": int(
        (
            ~ranked_queue["final_refresh_score"].between(0, 100)
        ).sum()
    ),
}

# 2. Score and queue distribution snapshot
score_monitoring = {
    "model_probability_mean": float(
        ranked_queue["best_model_probability"].mean()
    ),
    "model_probability_median": float(
        ranked_queue["best_model_probability"].median()
    ),
    "model_probability_p90": float(
        ranked_queue["best_model_probability"].quantile(0.90)
    ),
    "final_score_mean": float(
        ranked_queue["final_refresh_score"].mean()
    ),
    "final_score_median": float(
        ranked_queue["final_refresh_score"].median()
    ),
    "final_score_p90": float(
        ranked_queue["final_refresh_score"].quantile(0.90)
    ),
}

confidence_distribution = (
    ranked_queue["confidence"]
    .value_counts(normalize=True)
    .to_dict()
)

action_distribution = (
    ranked_queue["suggested_action"]
    .value_counts()
    .to_dict()
)

# 3. Practical triggers for human investigation
trigger_policy = {
    "data_quality": [
        "Investigate any duplicate content IDs.",
        "Investigate missing model probabilities or final scores.",
        "Review unexpected feature missingness or range changes.",
    ],
    "score_distribution": [
        "Review substantial changes in model probability or final-score distributions.",
        "Compare current distributions with a saved historical monitoring snapshot.",
    ],
    "ranking_usefulness": [
        "Review repeated low-value recommendations among the highest-ranked pages.",
        "Collect reviewer acceptance, rejection, and deferral rates.",
    ],
    "retraining": [
        "Consider retraining only after label and feature timing checks.",
        "Require enough new labelled examples.",
        "Re-evaluate using grouped or time-aware validation.",
        "Do not automatically replace the current model.",
    ],
    "important_caveat": (
        "This notebook creates a baseline monitoring snapshot. "
        "It does not establish statistical drift thresholds or run continuous monitoring."
    ),
}

monitoring_report = {
    "monitoring_checks": monitoring_checks,
    "score_distribution": score_monitoring,
    "confidence_distribution": confidence_distribution,
    "suggested_action_distribution": action_distribution,
    "trigger_policy": trigger_policy,
}

monitoring_path = WEEK7_OUTPUT_DIR / "monitoring_snapshot.json"

with open(monitoring_path, "w", encoding="utf-8") as f:
    json.dump(monitoring_report, f, indent=2)

print("Data-quality checks:")
display(
    pd.DataFrame(
        list(monitoring_checks.items()),
        columns=["Check", "Value"]
    )
)

print("\nScore distribution:")
display(
    pd.DataFrame(
        list(score_monitoring.items()),
        columns=["Metric", "Value"]
    ).round(4)
)

print("\nMonitoring snapshot saved to:", monitoring_path)
print(
    "\nNote: This is a one-time snapshot. "
    "A future monitoring run should compare it with a dated historical snapshot."
)

Data-quality checks:


,Check,Value
0,queue_rows,30000
1,unique_content_ids,30000
2,duplicate_content_ids,0
3,missing_model_probabilities,0
4,missing_final_scores,0
5,probabilities_outside_0_1,0
6,scores_outside_0_100,0



Score distribution:


,Metric,Value
0,model_probability_mean,0.5089
1,model_probability_median,0.5381
2,model_probability_p90,0.7238
3,final_score_mean,49.7995
4,final_score_median,53.6075
5,final_score_p90,68.1714



Monitoring snapshot saved to: /content/Flyrank-ML-Internship/work/outputs/monitoring_snapshot.json

Note: This is a one-time snapshot. A future monitoring run should compare it with a dated historical snapshot.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.